# Flash-Sale Concurrency & Telemetry Analysis
Phân tích dữ liệu độ trễ (Latency) và tính nhất quán dữ liệu (Data Consistency) giữa hai kịch bản **Naive SQLite** vs **Atomic Redis**.

In [ ]:
import sqlite3
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Đọc dữ liệu trực tiếp từ flashsale.db
db_path = os.path.join("..", "flashsale.db")
conn = sqlite3.connect(db_path)
df = pd.read_sql_query("SELECT * FROM telemetry_logs ORDER BY id ASC", conn)
conn.close()

print(f"Tổng số request đã ghi nhận: {len(df)}")
df.head()

## 1. Thống kê tổng quan theo Kịch bản

In [ ]:
summary = df.groupby(['mode', 'status']).agg(
    total_requests=('id', 'count'),
    latency_mean_ms=('latency_ms', 'mean'),
    latency_p50_ms=('latency_ms', lambda x: np.percentile(x, 50)),
    latency_p95_ms=('latency_ms', lambda x: np.percentile(x, 95)),
    latency_p99_ms=('latency_ms', lambda x: np.percentile(x, 99))
).reset_index()

summary

## 2. Trực quan hóa Phân bố Độ trễ (Latency Distribution)

In [ ]:
plt.style.use('dark_background')
plt.figure(figsize=(10, 5))

for mode, color in [('naive', '#f43f5e'), ('atomic', '#10b981')]:
    subset = df[df['mode'] == mode]
    if not subset.empty:
        plt.hist(subset['latency_ms'], bins=30, alpha=0.6, label=f'Mode: {mode.upper()}', color=color)

plt.title('So sánh Phân bố Độ trễ (Latency Distribution: Naive vs Atomic)', fontsize=12)
plt.xlabel('Latency (ms)')
plt.ylabel('Số lượng Request')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.2)
plt.show()

## 3. Phát hiện Bán Âm Vé (Race Condition vs Zero Oversell)

In [ ]:
plt.figure(figsize=(10, 5))

for mode, color in [('naive', '#f43f5e'), ('atomic', '#10b981')]:
    subset = df[(df['mode'] == mode) & (df['status'] == 'SUCCESS')].copy()
    if not subset.empty:
        subset['cumulative_sold'] = range(1, len(subset) + 1)
        time_rel = subset['created_at'] - subset['created_at'].min()
        plt.plot(time_rel, subset['cumulative_sold'], label=f'{mode.upper()} (Bán được: {len(subset)})', color=color, linewidth=2)

plt.axhline(y=100, color='#eab308', linestyle='--', label='Ngưỡng kho tối đa: 100 vé')
plt.title('Tích lũy Số Vé Bán Được Theo Thời Gian', fontsize=12)
plt.xlabel('Thời gian (giây)')
plt.ylabel('Số vé đã bán')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.2)
plt.show()